# Dip Adjusted Samples

This notebook will enable plotting of dip adjusted samples

## Import packages

In [20]:
# general
import numpy as np
from tqdm import tqdm
import pandas as pd

# plotting
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle
import matplotlib

# My packages
# my functions/classes
import sys
sys.path.append("../core_scripts/")
from ECMclass import ECM, core_section


## Set up

In [ ]:
# Set filepaths

path_to_data = '../../data/'
path_to_samples = '../../data/sampling/master/'
path_to_angles = '../../data/angles/'
path_to_figures = '../../../figures/dip_adjusted_samples/'

path_to_final_figures = '../../../paper/final_figures/'

: 

In [ ]:
ic_icpms_dict = {
    'Ca2+': 'Ca',
    'Mg+': 'Mg',
    'Na+': 'Na',
    'K+': 'K',
    #'SO42-': 'S',
}

global ic_icpms_dict

: 

## Load Data

In [ ]:
# load unit data
units = pd.read_excel(path_to_data+'sampling/units.xlsx')

: 

In [ ]:
# Load dip data
angles = pd.read_csv(path_to_angles + 'angles.csv')
angles.head()

: 

In [ ]:
# Load ECM data

# Load metadata
meta = pd.read_csv(path_to_data + 'ecm/alligned/metadata.csv')
meta.head()

# set smoothing window
window = 10

# Load ECM data
data = []
cores = []
sections = []
faces = []
ACorDCs = []
for index, row in tqdm(meta.iterrows(), total=len(meta), desc="Processing data"):
    
    core = row['core']
        
    section = row['section']
    face = row['face']
    ACorDC = row['ACorDC']

    if ACorDC == 'AC' and not face == 'o':

        data_item = ECM(core,section,face,ACorDC,path_to_data + 'ecm/alligned/')
        print("Reading "+core+", section "+section+'-'+face+'-'+ACorDC)
        
        data_item.rem_ends(15)
        data_item.smooth(window)
        data.append(data_item)
        
        cores.append(core)
        sections.append(section)
        faces.append(face)
        ACorDCs.append(ACorDC)
        

# Put ECM data into core_section structure
s228_4_AC = core_section('228_4','alhic1901','AC',data,sections,faces,cores,ACorDCs)
s230_4_AC = core_section('230_4','alhic1901','AC',data,sections,faces,cores,ACorDCs)

# place in 3D space
for s in [s228_4_AC,s230_4_AC]:
    s.add_3d_coords()

# add angles
for s in [s228_4_AC,s230_4_AC]:
    s.get_angles(angles)


: 

In [ ]:
# load sample data
samps = ['water_iso','cc','ic_icpms','cfa','ghg']

# loop through and load all sample files
for s in samps:

    if s == samps[0]:
        samp_data_raw = pd.read_csv(path_to_samples + s+'_master.csv')

    else:

        df = pd.read_csv(path_to_samples + s+'_master.csv')
        samp_data_raw = pd.concat([samp_data_raw, df], ignore_index=True)

: 

## Make some functions

In [ ]:
def plot_ecm(axs,face,top_angle,cmap,rescale):

    res = 0.001 #0.002

    meas = face.meas_s
    button = face.button_s
    ycor = face.y_s
    d = face.depth_s
    x_vector = face.x_3d
    y_vector = face.y_3d
    tracks = np.unique(x_vector)

    # define width of track
    width = tracks[2] - tracks[1]

    for x in tracks:
        
        # Pull out data for this track
        idx = x_vector==x
        tmeas = meas[idx]
        tbut = button[idx]
        td = d[idx]

        # calculate the shift
        depth_shift = x * np.tan(top_angle * np.pi/180)
        td = td + depth_shift

        # downsample ECM to save plotting time (as needed)
        if res != 0:
            int_lo = round(min(td),3)
            int_hi = round(max(td),3)
            depth_interp = np.linspace(int_lo,int_hi,int((int_hi-int_lo)/res)+1)
            meas_interp = np.interp(depth_interp,np.flip(td),np.flip(tmeas))
            but_interp = np.interp(depth_interp,np.flip(td),np.flip(tbut))
            td = depth_interp
            tmeas = meas_interp
            tbut = np.round(but_interp)
        
        for i in range(len(tmeas)-1):
            
            if tbut[i] == 0:
                axs.add_patch(Rectangle((x-(width)/2,td[i]),(width),td[i+1]-td[i],facecolor=cmap(rescale(tmeas[i]))))
            else:
                axs.add_patch(Rectangle((x-(width)/2,td[i]),(width),td[i+1]-td[i],facecolor=cmap(rescale(tmeas[i]))))

    

: 

In [ ]:
# plot sample
def plot_sample(ax0,axs,samp_data_all,top_angle,side_angle,cmap,p,proxies,section):

    linewidth = 0.8

    # # get the count
    # samp_data_allprox = samp_data_all[samp_data_all[proxies].notna().any(axis=1)]
    # sticks_all = samp_data_allprox['stick'].unique()

    # filter samp_data for only rows where the column p is not empty
    samp_data_all = samp_data_all[samp_data_all[p].notna()]

    #get unique values in column 'sticks'
    sticks = samp_data_all['stick'].unique()

    # filter samp_data for only rows where 'section' is equal to sec.section
    samp_data = samp_data_all[samp_data_all['section'] == section]

    # loop through each unique value in column 'sticks'
    for s in sticks:

        # filter samp_data for only rows where the column 'sticks' is equal to s
        df = samp_data[samp_data['stick'] == s].copy()

        # compute the shift
        shift = df['effective_center_x'] * np.tan(top_angle * np.pi/180) + df['effective_center_y'] * np.tan(side_angle * np.pi/180)
        df['mid_depth_shifted'] = df['mid_depth'] + shift
        df['top_depth_shifted'] = df['top_depth'] + shift
        df['bottom_depth_shifted'] = df['bottom_depth'] + shift

        # compute shifts for four corners of the cube
        left_shift = df['x_lo'] * np.tan(top_angle * np.pi/180) + df['effective_center_y'] * np.tan(side_angle * np.pi/180)
        right_shift =  df['x_hi'] * np.tan(top_angle * np.pi/180) + df['effective_center_y'] * np.tan(side_angle * np.pi/180)

        df['top_depth_left'] = df['top_depth'] + left_shift
        df['top_depth_right'] = df['top_depth'] + right_shift
        df['bottom_depth_left'] = df['bottom_depth'] + left_shift
        df['bottom_depth_right'] = df['bottom_depth'] + right_shift

        # get the color
        if not df.empty:

            color = cmap(int(df['default_color'].mean()))

            # plot the line
            axs.plot(df[p],df['mid_depth_shifted'],color=color,linewidth=linewidth)

            for index,row in df.iterrows():

                # plot the cube
                ax0.plot([row['x_lo'],row['x_hi']],[row['top_depth_left'],row['top_depth_right']],color=color,linewidth=linewidth)
                ax0.plot([row['x_lo'],row['x_hi']],[row['bottom_depth_left'],row['bottom_depth_right']],color=color,linewidth=linewidth)
                ax0.plot([row['x_lo'],row['x_lo']],[row['bottom_depth_left'],row['top_depth_left']],color=color,linewidth=linewidth)
                ax0.plot([row['x_hi'],row['x_hi']],[row['bottom_depth_right'],row['top_depth_right']],color=color,linewidth=linewidth)
                # ax0.plot([row['x_lo'],row['x_hi'],row['x_hi'],row['x_lo'],row['x_lo']],
                #          [row['top_depth_right'],row['top_depth_right'],row['bottom_depth_left'],row['bottom_depth_left'],row['top_depth_right']],
                #          color=color)

                # plot the line
                axs.plot([row[p],row[p]],[row['top_depth_shifted'],row['bottom_depth_shifted']],color=color,linewidth=linewidth)


: 

In [ ]:

def make_full_plot(samp_data,secs,proxies,side_angles,top_angles,d_ranges,title=None):

    # set colormap - ecm
    my_cmap = matplotlib.colormaps['Spectral']

    # set colormap - lines
    cmap_line = matplotlib.colormaps['tab10']

    # filter samp_data for only rows where 'section' is in the list of sections from secs
    sections = [sec.section for sec in secs]
    samp_data_sec = samp_data[samp_data['section'].isin(sections)]

    # define rescale
    meas_all = np.array([])
    for sec in secs:
        meas_all = np.concatenate((meas_all,sec.top.meas_s))
    pltmin = np.percentile(meas_all,5)
    pltmax = np.percentile(meas_all,95)
    rescale = lambda k: (k-pltmin) /  (pltmax-pltmin)

    # make figure - figure out width ratios
    ratios = [5]
    numplots=2
    if len(proxies)>0:
        numplots=1+len(proxies)
    for i in range(len(proxies)):
        ratios.append(2.5)

    #make figure - figure out height ratios
    h_ratios = []
    for d in d_ranges:
        h_ratios.append(abs(d[1]-d[0]))
    ave = sum(h_ratios)/len(h_ratios)
    h_ratios = [x/ave for x in h_ratios]
    

    # make figure
    #fig, ax = plt.subplots(len(secs), numplots, figsize=(7+len(proxies)*3, 6*len(d_ranges)), gridspec_kw={'width_ratios': ratios, 'height_ratios': h_ratios}, dpi=100)
    fig, ax = plt.subplots(len(secs), numplots, figsize=(7,7), gridspec_kw={'width_ratios': ratios, 'height_ratios': h_ratios}, dpi=300)

    # add subplot letters to the top row
    for i in range(numplots):
        ax[0,i].text(0.02, 0.98, f"({chr(97+i)})", transform=ax[0,i].transAxes,
                     va='top', ha='left', fontsize=8)

    # ecm subplot admin
    ax[0,0].set_xlim([-.120,.120])
    ax[len(secs)-1,0].set_xlabel('Distance across core (m)',fontsize=8)
    ax[0,0].set_title('ECM and Sample Locations', fontsize=8)    
    ax[0,0].set_xticklabels([])
    ax[0,0].set_xticks([])
    for i in range(len(secs)):
        ax[i,0].yaxis.set_major_locator(plt.MultipleLocator(0.1))

    cnt = 0
    for sec, top_angle, side_angle, d_range in zip(secs, top_angles, side_angles, d_ranges):

        # set top and bottom depths
        #td = min(sec.top.depth_s) - 0.12
        #bd = max(sec.top.depth_s) + 0.12
        td = d_range[1]
        bd = d_range[0]

        # set axis limits
        for axs in ax[cnt,:]:
            axs.set_ylim([bd,td])

        # plot ECM data
        plot_ecm(ax[cnt,0],sec.top,top_angle,my_cmap,rescale=rescale)

        # loop through proxies
        for i in range(len(proxies)):

            # pull out values
            p = proxies[i]
            axs = ax[cnt,i+1]

            # plot the proxy data (c_cnt keeps tracks of which colors have been used)
            plot_sample(ax[cnt,0],axs,samp_data_sec,top_angle,side_angle,cmap_line,p,proxies,sec.section)

            # set figure axis limits
            prox_data_min = samp_data_sec[p].min()
            prox_data_max = samp_data_sec[p].max()

            # look up fancy name and units
            unit_row = units.loc[units['simple_name'] == p]
            if len(unit_row) == 1:
                full_name = unit_row.latex_name.iloc[0]
                unit = ' ('+unit_row.units.iloc[0]+')'
            else:
                full_name = p
                unit = ''
            xlabel = full_name + unit

            # flag ic proxies to also plot icpms
            if proxies[i] in list(ic_icpms_dict.keys()):

                # plot
                plot_sample(ax[cnt,0],axs,samp_data_sec,top_angle,side_angle,cmap_line,ic_icpms_dict[proxies[i]],proxies,sec.section)

                # update min and max
                prox_data_min = min(prox_data_min, samp_data_sec[ic_icpms_dict[proxies[i]]].min())
                prox_data_max = max(prox_data_max, samp_data_sec[ic_icpms_dict[proxies[i]]].max())

                # combine full names
                unit_row_icpms = units.loc[units['simple_name'] == ic_icpms_dict[proxies[i]]]
                if len(unit_row_icpms) == 1:
                    full_name2 = unit_row_icpms.latex_name.iloc[0]
                    unit2 = unit_row_icpms.units.iloc[0]
                else:
                    full_name2 =  ic_icpms_dict[proxies[i]]
                    unit2 = ''
                # set x-label, dealing with special case of ppb/ug/L
                xlabel = full_name2  + unit
                # if unit2 in ['µg/L','ppb'] and unit in ['µg/L','ppb']:
                #     xlabel = full_name + ' and ' + full_name2 + ' (' + unit + ')'
                # else:
                #     xlabel = xlabel + ' and ' + full_name2 + ' (' + unit2 + ')'

            # figure axis limits
            cushion = abs(prox_data_max - prox_data_min) * 0.05
            axs.set_xlim([prox_data_min-cushion,prox_data_max+cushion])

            # axis housekeeping - deleting all in this case
            if i==len(proxies)-1: # if last one
                # axs.yaxis.set_label_position('right')
                # axs.yaxis.tick_right()

                # set y-tick label font size to be 8
                #axs.tick_params(labelsize=8)
                axs.set_yticklabels([])

            else:
                axs.set_yticklabels([]) 
            axs.yaxis.set_major_locator(plt.MultipleLocator(0.1))

            #set tick label font size on all axis
            axs.tick_params(axis='both', which='major', labelsize=8)

            # set axis labels
            if cnt == 0:
                axs.set_xlabel(xlabel,fontsize=8)
                axs.xaxis.set_label_position('top')
                axs.xaxis.tick_top()
            if cnt == len(secs)-1:
                axs.set_xlabel(xlabel,fontsize=8)

        for a in ax[cnt,:]:
            a.grid(axis='y')
                
        # increment counter
        cnt += 1

    #fig.tight_layout()
    # add y-axis labels
    fig.text(0.04, 0.5, 'Dip-Adjusted Depth (m)', va='center', rotation='vertical', fontsize=8)
    #fig.text(1, 0.5, 'Dip-Adjusted Depth (m)', va='center', rotation='vertical', fontsize=8)
    fig.tight_layout(rect=[0.05, 0, 1, 0.95])


    plt.subplots_adjust(wspace=0)
    plt.subplots_adjust(hspace=0.01)



    # add colorbar (width fixed to first two subplots)
    cbar_ax = fig.add_axes([ax[0,0].get_position().x0,-0.03,ax[0,0].get_position().x1-ax[0,0].get_position().x0,0.03])
    norm = matplotlib.colors.Normalize(vmin=pltmin,vmax=pltmax)
    cbar = fig.colorbar(matplotlib.cm.ScalarMappable(norm=norm, cmap=my_cmap),cax=cbar_ax,
                    orientation='horizontal',label='AC Current (amps)')
    cbar.ax.tick_params(labelsize=8)
    cbar.set_label('AC Current (amps)', fontsize=8)
    
    # save plot
    plt.savefig(
        path_to_figures 
        + f"Combined{'_'.join(proxies)}_top{top_angle:.2f}_side{side_angle:.2f}.png",
        bbox_inches='tight',
        dpi=300
    )

    return fig, ax

: 

In [ ]:
# define function inputs (temporary)
sec = s228_4_AC
proxies = ['d18O','co2']
side_angle = None
top_angle = None

: 

## Now let's make some plots!!!

In [ ]:
secs = [s228_4_AC,s230_4_AC]
proxies = ['Cl-','Ca2+','PC1']
d_range = [[155.45, 154.95],[157.15,156.75]]
side_angles = [s228_4_AC.side_angle,s230_4_AC.side_angle]
top_angles = [s228_4_AC.top_angle,s230_4_AC.top_angle]
fig, ax = make_full_plot(samp_data_raw,secs,proxies,side_angles,top_angles,d_range,title=None)

# save final figure as pdf and png
fig.savefig(path_to_final_figures+'pdf/chem_results.pdf', dpi=300)
fig.savefig(path_to_final_figures+'png/chem_results.png', dpi=300)

: 

In [ ]:
secs = [s228_4_AC,s230_4_AC]
proxies = ['d18O','co2','ch4']
d_range = [[155.45, 154.95],[157.15,156.75]]
side_angles = [s228_4_AC.side_angle,s230_4_AC.side_angle]
top_angles = [s228_4_AC.top_angle,s230_4_AC.top_angle]
fig, ax = make_full_plot(samp_data_raw,secs,proxies,side_angles,top_angles,d_range,title=None)

# save final figure as pdf and png
fig.savefig(path_to_final_figures+'pdf/ghgs.pdf', dpi=300)
fig.savefig(path_to_final_figures+'png/ghgs.png', dpi=300)

: 

In [ ]:
secs = [s228_4_AC,s230_4_AC]
proxies = ['Na','SO42-','S']
d_range = [[155.45, 154.95],[157.15,156.75]]
side_angles = [s228_4_AC.side_angle,s230_4_AC.side_angle]
top_angles = [s228_4_AC.top_angle,s230_4_AC.top_angle]
fig, ax = make_full_plot(samp_data_raw,secs,proxies,side_angles,top_angles,d_range,title=None)

# save final figure as pdf and png
fig.savefig(path_to_final_figures+'pdf/extra_chem.pdf', dpi=300)
fig.savefig(path_to_final_figures+'png/extra_chem.png', dpi=300)

: 

: 

: 

: 

: 